In [1]:
import numpy as np
from scipy.spatial.distance import jensenshannon
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
import pandas as pd


# -------------------------
# 1. Check probability validity
# -------------------------
def s_nan(y_prob, eps=1e-3):
    if not np.all(np.isfinite(y_prob)):
        return 0

    row_sums = np.sum(y_prob, axis=1)
    if np.all(np.abs(row_sums - 1.0) <= eps):
        return 1
    return 0


# -------------------------
# 2. Majority + JSD score
# -------------------------
def s_maj_jsd(y_true, y_pred, n_classes=None):
    if n_classes is None:
        n_classes = len(np.unique(y_true))

    pred_dist = np.bincount(y_pred, minlength=n_classes) / len(y_pred)
    true_dist = np.bincount(y_true, minlength=n_classes) / len(y_true)

    # Majority score
    f_max = np.max(pred_dist)
    s_maj = 1 - (f_max - 1/n_classes) / (1 - 1/n_classes)

    # Jensen-Shannon divergence
    # jensenshannon() trả về căn bậc hai của JSD dùng log e
    js_distance = jensenshannon(true_dist, pred_dist)
    
    # Bình phương để lấy JSD (Divergence)
    js_divergence = js_distance**2
    s_jsd = 1 - (js_divergence / np.log(2))

    return s_maj, s_jsd


# -------------------------
# 3. Entropy score
# -------------------------
def s_ent(y_prob):
    N, K = y_prob.shape

    h = -np.sum(y_prob * np.log(y_prob + 1e-10), axis=1) / np.log(K)
    m = np.median(h)

    return 1 - min(abs(m - 0.5) / 0.5, 1.0)


# -------------------------
# 4. Drift score
# -------------------------
def s_drift(y_pred, ref_dist=None, n_classes=None):
    if n_classes is None:
        n_classes = len(np.unique(y_pred))

    pred_dist = np.bincount(y_pred, minlength=n_classes) / len(y_pred)

    if ref_dist is None:
        ref_dist = np.ones(n_classes) / n_classes

    # jensenshannon trả về căn bậc hai của JSD (Distance)
    js_distance = jensenshannon(pred_dist, ref_dist)
    
    # Bình phương để có JSD (Divergence)
    js_divergence = js_distance**2
    
    return 1 - (js_divergence / np.log(2))


# -------------------------
# 5. Efficiency score
# -------------------------
def s_eff(mask):
    return np.mean(mask)


# -------------------------
# 6. Leakage score (generic probe)
# -------------------------
def s_leak_1(mask, y_true, probe_model, test_size=0.3, random_state=40):

    X_train, X_test, y_train, y_test = train_test_split(
        mask, y_true,
        test_size=test_size,
        random_state=random_state,
        stratify=y_true
    )

    probe_model.fit(X_train, y_train)

    if hasattr(probe_model, "predict_proba"):
        y_score = probe_model.predict_proba(X_test)
    else:
        y_score = probe_model.decision_function(X_test)

    auc = roc_auc_score(y_test, y_score, multi_class="ovr")
    return 1 - auc


# -------------------------
# 7. Logistic leakage
# -------------------------
def s_leak(X, y, test_size=0.3, random_state=40):

    Xtr, Xte, ytr, yte = train_test_split(
        X, y,
        test_size=test_size,
        stratify=y,
        random_state=random_state
    )

    probe = LogisticRegression(max_iter=1000)
    probe.fit(Xtr, ytr)

    y_score = probe.predict_proba(Xte)

    auc = roc_auc_score(yte, y_score, multi_class="ovr")
    return float(1 - auc)

# -------------------------
# 8. Final geometric mean score
# -------------------------
def S_san_plus(values):
    values = np.clip(values, 1e-10, 1)
    return np.prod(values) ** (1 / len(values))


In [2]:
def compute_ref_dist(y_train):
    y = y_train.astype(int).to_numpy()
    n_classes = len(np.unique(y))
    return np.bincount(y, minlength=n_classes) / len(y)

def run_sanity_evaluation(
    *,
    path_1,
    path_2,
    version_name,
    train_name,
    n_classes=3
):
    print(
        f"[START] Sanity evaluation | "
        f"train={train_name} | version={version_name}"
    )

    rows = []

    # -----------------
    # Load train
    # -----------------
    train_df = pd.read_csv(f"{path_1}/{train_name}.csv")
    y_train = train_df["label_3"]
    ref_dist = compute_ref_dist(y_train)

    # -----------------
    # Loop phases
    # -----------------
    for phase in [1, 2, 3, 4]:
        print(f"  -> Running phase {phase}...")

        # ---- probability matrix
        prob_df = pd.read_csv(
            f"{path_2}/probability_matrix_{version_name}_phase{phase}.csv"
        )

        y_true = prob_df["y_true"].astype(int).to_numpy()
        y_pred = prob_df["y_pred"].astype(int).to_numpy()
        y_prob = prob_df[
            ["Prob_Class_0", "Prob_Class_1", "Prob_Class_2"]
        ].to_numpy()

        # ---- test data (for leakage & efficiency)
        test_df = pd.read_csv(f"{path_1}/test_{phase}.csv")
        X_test = test_df.drop(columns=["label_3"], errors="ignore")
        mask = X_test.notna().astype(int).values

        # ---- metrics
        smaj, sjsd = s_maj_jsd(y_true, y_pred, n_classes=n_classes)

        snan   = s_nan(y_prob)
        sent   = s_ent(y_prob)
        sdrift = s_drift(y_pred, ref_dist, n_classes=n_classes)
        sleak  = s_leak(mask, y_true)
        s_eff_ = s_eff(mask)

        s_san_plus = S_san_plus([
            snan,
            sjsd,
            sent,
            sdrift,
            s_eff_,
            sleak
        ])

        rows.append({
            "train_name": train_name,
            "version_name": version_name,
            "phase": phase,
            "snan": snan,
            "smaj": smaj,
            "sjsd": sjsd,
            "sent": sent,
            "sdrift": sdrift,
            "sleak": sleak,
            "s_san+": s_san_plus,
            "s_eff": s_eff_
        })

    print(
        f"[END] Sanity evaluation DONE | "
        f"rows={len(rows)} | "
        f"phases=4"
    )

    columns_order = [
        "train_name",
        "version_name",
        "phase",
        "snan",
        "smaj",
        "sjsd",
        "sent",
        "sdrift",
        "sleak",
        "s_san+",
        "s_eff"
    ]

    return pd.DataFrame(rows)[columns_order]


# Chạy với các version data

## V1 (Median)

In [3]:
path_1 = "/kaggle/input/lo-dataset/Median/Median"

In [4]:
path_2 = "/kaggle/input/datasets/anhtran10/lightgbm-results/LightGBM_results"

In [5]:
df_v1 = run_sanity_evaluation(
    path_1=path_1,
    path_2=path_2,
    version_name="V1 (Median)",
    train_name="train_median"
)
df_v1

[START] Sanity evaluation | train=train_median | version=V1 (Median)
  -> Running phase 1...
  -> Running phase 2...
  -> Running phase 3...
  -> Running phase 4...
[END] Sanity evaluation DONE | rows=4 | phases=4


,train_name,version_name,phase,snan,smaj,sjsd,sent,sdrift,sleak,s_san+,s_eff
0,train_median,V1 (Median),1,1,0.000000,0.998217,0.000000,0.998217,0.5,0.019182,1.0
1,train_median,V1 (Median),2,1,0.000123,0.998440,0.000000,0.998440,0.5,0.019184,1.0
2,train_median,V1 (Median),3,1,0.145604,0.954566,0.000000,0.954565,0.5,0.018899,1.0
3,train_median,V1 (Median),4,1,0.005556,0.999990,0.000115,0.999990,0.5,0.196481,1.0


In [6]:
df_v1.to_csv("results_v1.csv", index=False)

## V2 (Median CDSMOTE)

In [7]:
df_v2 = run_sanity_evaluation(
    path_1=path_1,
    path_2=path_2,
    version_name="V2 (Median CDSMOTE)",
    train_name="train_median_cdsmote"
)
df_v2

[START] Sanity evaluation | train=train_median_cdsmote | version=V2 (Median CDSMOTE)
  -> Running phase 1...
  -> Running phase 2...
  -> Running phase 3...
  -> Running phase 4...
[END] Sanity evaluation DONE | rows=4 | phases=4


,train_name,version_name,phase,snan,smaj,sjsd,sent,sdrift,sleak,s_san+,s_eff
0,train_median_cdsmote,V2 (Median CDSMOTE),1,1,0.000000,0.998217,0.002903,0.540852,0.5,0.303634,1.0
1,train_median_cdsmote,V2 (Median CDSMOTE),2,1,0.000245,0.998619,0.002873,0.541899,0.5,0.303236,1.0
2,train_median_cdsmote,V2 (Median CDSMOTE),3,1,0.005298,0.999846,0.015159,0.556023,0.5,0.401892,1.0
3,train_median_cdsmote,V2 (Median CDSMOTE),4,1,0.008802,0.999786,0.002639,0.563612,0.5,0.300983,1.0


In [8]:
df_v2.to_csv("results_v2.csv", index=False)

## V3 (Median SASMOTE)

In [9]:
df_v3 = run_sanity_evaluation(
    path_1=path_1,
    path_2=path_2,
    version_name="V3 (Median SASMOTE)",
    train_name="train_median_sasmote"
)
df_v3

[START] Sanity evaluation | train=train_median_sasmote | version=V3 (Median SASMOTE)
  -> Running phase 1...
  -> Running phase 2...
  -> Running phase 3...
  -> Running phase 4...
[END] Sanity evaluation DONE | rows=4 | phases=4


,train_name,version_name,phase,snan,smaj,sjsd,sent,sdrift,sleak,s_san+,s_eff
0,train_median_sasmote,V3 (Median SASMOTE),1,1,0.000000,0.998217,0.011551,0.540852,0.5,0.382219,1.0
1,train_median_sasmote,V3 (Median SASMOTE),2,1,0.000458,0.998825,0.017716,0.542640,0.5,0.410729,1.0
2,train_median_sasmote,V3 (Median SASMOTE),3,1,0.005227,0.999838,0.015721,0.555853,0.5,0.404317,1.0
3,train_median_sasmote,V3 (Median SASMOTE),4,1,0.008937,0.999770,0.002709,0.563913,0.5,0.302332,1.0


In [10]:
df_v3.to_csv("results_v3.csv", index=False)

## V4 (Median RadiusSMOTE)

In [11]:
df_v4 = run_sanity_evaluation(
    path_1=path_1,
    path_2=path_2,
    version_name="V4 (Median RadiusSMOTE)",
    train_name="train_median_radiussmote"
)
df_v4

[START] Sanity evaluation | train=train_median_radiussmote | version=V4 (Median RadiusSMOTE)
  -> Running phase 1...
  -> Running phase 2...
  -> Running phase 3...
  -> Running phase 4...
[END] Sanity evaluation DONE | rows=4 | phases=4


,train_name,version_name,phase,snan,smaj,sjsd,sent,sdrift,sleak,s_san+,s_eff
0,train_median_radiussmote,V4 (Median RadiusSMOTE),1,1,0.000529,0.998834,0.018705,0.542794,0.5,0.414484,1.0
1,train_median_radiussmote,V4 (Median RadiusSMOTE),2,1,0.001865,0.999381,0.026555,0.546559,0.5,0.439960,1.0
2,train_median_radiussmote,V4 (Median RadiusSMOTE),3,1,0.006143,0.999949,0.026303,0.557327,0.5,0.440735,1.0
3,train_median_radiussmote,V4 (Median RadiusSMOTE),4,1,0.006511,0.999967,0.001689,0.558614,0.5,0.279007,1.0


In [12]:
df_v4.to_csv("results_v4.csv", index=False)

## V5 (Mean)

In [13]:
path_1 = "/kaggle/input/lo-dataset/Mean/Mean"

In [14]:
df_v5 = run_sanity_evaluation(
    path_1=path_1,
    path_2=path_2,
    version_name="V5 (Mean)",
    train_name="train_mean"
)
df_v5

[START] Sanity evaluation | train=train_mean | version=V5 (Mean)
  -> Running phase 1...
  -> Running phase 2...
  -> Running phase 3...
  -> Running phase 4...
[END] Sanity evaluation DONE | rows=4 | phases=4


,train_name,version_name,phase,snan,smaj,sjsd,sent,sdrift,sleak,s_san+,s_eff
0,train_mean,V5 (Mean),1,1,0.016016,0.997985,0.000000,0.997984,0.5,0.019181,1.0
1,train_mean,V5 (Mean),2,1,0.011351,0.999462,0.000000,0.999462,0.5,0.019190,1.0
2,train_mean,V5 (Mean),3,1,0.008653,0.999661,0.000000,0.999661,0.5,0.019192,1.0
3,train_mean,V5 (Mean),4,1,0.005943,0.999987,0.000002,0.999987,0.5,0.096204,1.0


In [15]:
df_v5.to_csv("results_v5.csv", index=False)

## V6 (Mean CDSMOTE)

In [16]:
df_v6 = run_sanity_evaluation(
    path_1=path_1,
    path_2=path_2,
    version_name="V6 (Mean CDSMOTE)",
    train_name="train_mean_cdsmote"
)
df_v6

[START] Sanity evaluation | train=train_mean_cdsmote | version=V6 (Mean CDSMOTE)
  -> Running phase 1...
  -> Running phase 2...
  -> Running phase 3...
  -> Running phase 4...
[END] Sanity evaluation DONE | rows=4 | phases=4


,train_name,version_name,phase,snan,smaj,sjsd,sent,sdrift,sleak,s_san+,s_eff
0,train_mean_cdsmote,V6 (Mean CDSMOTE),1,1,0.000000,0.998217,0.005340,0.540852,0.5,0.336102,1.0
1,train_mean_cdsmote,V6 (Mean CDSMOTE),2,1,0.000026,0.998283,0.008755,0.540983,0.5,0.364990,1.0
2,train_mean_cdsmote,V6 (Mean CDSMOTE),3,1,0.003304,0.999589,0.016864,0.550992,0.5,0.408458,1.0
3,train_mean_cdsmote,V6 (Mean CDSMOTE),4,1,0.008937,0.999766,0.003174,0.563945,0.5,0.310415,1.0


In [17]:
df_v6.to_csv("results_v6.csv", index=False)

## V7 (Mean SASMOTE)

In [18]:
df_v7 = run_sanity_evaluation(
    path_1=path_1,
    path_2=path_2,
    version_name="V7 (Mean SASMOTE)",
    train_name="train_mean_sasmote"
)
df_v7

[START] Sanity evaluation | train=train_mean_sasmote | version=V7 (Mean SASMOTE)
  -> Running phase 1...
  -> Running phase 2...
  -> Running phase 3...
  -> Running phase 4...
[END] Sanity evaluation DONE | rows=4 | phases=4


,train_name,version_name,phase,snan,smaj,sjsd,sent,sdrift,sleak,s_san+,s_eff
0,train_mean_sasmote,V7 (Mean SASMOTE),1,1,0.000000,0.998217,0.003388,0.540852,0.5,0.311561,1.0
1,train_mean_sasmote,V7 (Mean SASMOTE),2,1,0.000045,0.998326,0.012236,0.541079,0.5,0.385944,1.0
2,train_mean_sasmote,V7 (Mean SASMOTE),3,1,0.003517,0.999599,0.018257,0.551537,0.5,0.413966,1.0
3,train_mean_sasmote,V7 (Mean SASMOTE),4,1,0.009092,0.999749,0.003264,0.564273,0.5,0.311909,1.0


In [19]:
df_v7.to_csv("results_v7.csv", index=False)

## V8 (Mean RadiusSMOTE)

In [20]:
df_v8 = run_sanity_evaluation(
    path_1=path_1,
    path_2=path_2,
    version_name="V8 (Mean RadiusSMOTE)",
    train_name="train_mean_radiussmote"
)
df_v8

[START] Sanity evaluation | train=train_mean_radiussmote | version=V8 (Mean RadiusSMOTE)
  -> Running phase 1...
  -> Running phase 2...
  -> Running phase 3...
  -> Running phase 4...
[END] Sanity evaluation DONE | rows=4 | phases=4


,train_name,version_name,phase,snan,smaj,sjsd,sent,sdrift,sleak,s_san+,s_eff
0,train_mean_radiussmote,V8 (Mean RadiusSMOTE),1,1,0.000000,0.998217,0.005417,0.540852,0.5,0.336905,1.0
1,train_mean_radiussmote,V8 (Mean RadiusSMOTE),2,1,0.000013,0.998254,0.015456,0.540922,0.5,0.401245,1.0
2,train_mean_radiussmote,V8 (Mean RadiusSMOTE),3,1,0.002994,0.999821,0.029872,0.550190,0.5,0.449205,1.0
3,train_mean_radiussmote,V8 (Mean RadiusSMOTE),4,1,0.007511,0.999910,0.002499,0.560779,0.5,0.298017,1.0


In [21]:
df_v8.to_csv("results_v8.csv", index=False)

## V9 (Extra Trees)

In [22]:
path_1 = "/kaggle/input/lo-dataset/Extra_trees/Extra_trees"

In [23]:
df_v9 = run_sanity_evaluation(
    path_1=path_1,
    path_2=path_2,
    version_name="V9 (MissForest)",
    train_name="train_extra"
)
df_v9

[START] Sanity evaluation | train=train_extra | version=V9 (MissForest)
  -> Running phase 1...
  -> Running phase 2...
  -> Running phase 3...
  -> Running phase 4...
[END] Sanity evaluation DONE | rows=4 | phases=4


,train_name,version_name,phase,snan,smaj,sjsd,sent,sdrift,sleak,s_san+,s_eff
0,train_extra,V9 (MissForest),1,1,0.000510,0.998807,0.000000,0.998807,0.5,0.019186,1.0
1,train_extra,V9 (MissForest),2,1,0.025773,0.995150,0.000000,0.995150,0.5,0.019163,1.0
2,train_extra,V9 (MissForest),3,1,0.074867,0.982307,0.000070,0.982306,0.5,0.179596,1.0
3,train_extra,V9 (MissForest),4,1,0.005582,0.999989,0.000056,0.999989,0.5,0.174034,1.0


In [24]:
df_v9.to_csv("results_v9.csv", index=False)

## V10 (Extra Trees CDSMOTE)

In [25]:
df_v10 = run_sanity_evaluation(
    path_1=path_1,
    path_2=path_2,
    version_name="V10 (MissForest CDSMOTE)",
    train_name="train_extra_cdsmote"
)
df_v10

[START] Sanity evaluation | train=train_extra_cdsmote | version=V10 (MissForest CDSMOTE)
  -> Running phase 1...
  -> Running phase 2...
  -> Running phase 3...
  -> Running phase 4...
[END] Sanity evaluation DONE | rows=4 | phases=4


,train_name,version_name,phase,snan,smaj,sjsd,sent,sdrift,sleak,s_san+,s_eff
0,train_extra_cdsmote,V10 (MissForest CDSMOTE),1,1,0.000000,0.998217,0.000958,0.540852,0.5,0.252406,1.0
1,train_extra_cdsmote,V10 (MissForest CDSMOTE),2,1,0.000387,0.998713,0.001392,0.542319,0.5,0.268781,1.0
2,train_extra_cdsmote,V10 (MissForest CDSMOTE),3,1,0.001020,0.999100,0.011986,0.544258,0.5,0.385042,1.0
3,train_extra_cdsmote,V10 (MissForest CDSMOTE),4,1,0.008098,0.999855,0.001753,0.562124,0.5,0.281041,1.0


In [26]:
df_v10.to_csv("results_v10.csv", index=False)

## V11 (Extra Trees SASMOTE)

In [27]:
df_v11 = run_sanity_evaluation(
    path_1=path_1,
    path_2=path_2,
    version_name="V11 (MissForest SASMOTE)",
    train_name="train_extra_sasmote"
)
df_v11

[START] Sanity evaluation | train=train_extra_sasmote | version=V11 (MissForest SASMOTE)
  -> Running phase 1...
  -> Running phase 2...
  -> Running phase 3...
  -> Running phase 4...
[END] Sanity evaluation DONE | rows=4 | phases=4


,train_name,version_name,phase,snan,smaj,sjsd,sent,sdrift,sleak,s_san+,s_eff
0,train_extra_sasmote,V11 (MissForest SASMOTE),1,1,0.000000,0.998217,0.002041,0.540852,0.5,0.286324,1.0
1,train_extra_sasmote,V11 (MissForest SASMOTE),2,1,0.000000,0.998217,0.002285,0.540852,0.5,0.291769,1.0
2,train_extra_sasmote,V11 (MissForest SASMOTE),3,1,0.000323,0.998670,0.012718,0.542118,0.5,0.388584,1.0
3,train_extra_sasmote,V11 (MissForest SASMOTE),4,1,0.008227,0.999846,0.001778,0.562373,0.5,0.281716,1.0


In [28]:
df_v11.to_csv("results_v11.csv", index=False)

## V12 (Extra Trees RadiusSMOTE)

In [29]:
df_v12 = run_sanity_evaluation(
    path_1=path_1,
    path_2=path_2,
    version_name="V12 (MissForest RadiusSMOTE)",
    train_name="train_extra_radiussmote"
)
df_v12

[START] Sanity evaluation | train=train_extra_radiussmote | version=V12 (MissForest RadiusSMOTE)
  -> Running phase 1...
  -> Running phase 2...
  -> Running phase 3...
  -> Running phase 4...
[END] Sanity evaluation DONE | rows=4 | phases=4


,train_name,version_name,phase,snan,smaj,sjsd,sent,sdrift,sleak,s_san+,s_eff
0,train_extra_radiussmote,V12 (MissForest RadiusSMOTE),1,1,0.001433,0.999233,0.141106,0.545377,0.5,0.580960,1.0
1,train_extra_radiussmote,V12 (MissForest RadiusSMOTE),2,1,0.000478,0.998784,0.066928,0.542613,0.5,0.512573,1.0
2,train_extra_radiussmote,V12 (MissForest RadiusSMOTE),3,1,0.003188,0.999758,0.038836,0.550151,0.5,0.469278,1.0
3,train_extra_radiussmote,V12 (MissForest RadiusSMOTE),4,1,0.006156,0.999979,0.001579,0.557831,0.5,0.275820,1.0


In [30]:
df_v12.to_csv("results_v12.csv", index=False)